# 01 - Exploratory Data Analysis (EDA)

Loading and exploring the CIC-DDoS2019 dataset for multi-class classification.

In [1]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

sns.set_style("whitegrid")
%matplotlib inline

## Load Data (with sampling)

In [2]:
# Configuration
DATA_DIR = "../data/raw/CSVs"

# Find all CSV files
csv_files = glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True)
print(f"Found {len(csv_files)} CSV files")
for f in csv_files:
    print(f"  - {f}")

Found 18 CSV files
  - ../data/raw/CSVs/01-12/DrDoS_NTP.csv
  - ../data/raw/CSVs/01-12/Syn.csv
  - ../data/raw/CSVs/01-12/DrDoS_DNS.csv
  - ../data/raw/CSVs/01-12/TFTP.csv
  - ../data/raw/CSVs/01-12/UDPLag.csv
  - ../data/raw/CSVs/01-12/DrDoS_SSDP.csv
  - ../data/raw/CSVs/01-12/DrDoS_NetBIOS.csv
  - ../data/raw/CSVs/01-12/DrDoS_MSSQL.csv
  - ../data/raw/CSVs/01-12/DrDoS_UDP.csv
  - ../data/raw/CSVs/01-12/DrDoS_LDAP.csv
  - ../data/raw/CSVs/01-12/DrDoS_SNMP.csv
  - ../data/raw/CSVs/03-11/Syn.csv
  - ../data/raw/CSVs/03-11/MSSQL.csv
  - ../data/raw/CSVs/03-11/UDPLag.csv
  - ../data/raw/CSVs/03-11/LDAP.csv
  - ../data/raw/CSVs/03-11/UDP.csv
  - ../data/raw/CSVs/03-11/Portmap.csv
  - ../data/raw/CSVs/03-11/NetBIOS.csv


In [3]:

SAMPLE_ROWS = 0  # rows loaded from CSV file, 0 if all
RANDOM_STATE = 42
csv_file = '../data/raw/CSVs/01-12/DrDoS_NTP.csv'

# Load and sample data
def load_and_sample(filepath, n_rows=SAMPLE_ROWS, random_state=RANDOM_STATE):
    """Load CSV with optional sampling."""
    df = pd.read_csv(filepath)
    if len(df) > n_rows > 0:
        return df.sample(n=n_rows, random_state=random_state)
    return df


print(f"Loading {os.path.basename(csv_file)}...")
df = load_and_sample(f)
print(f"  -> {len(df)} rows")

# # Load all files
# dfs = []
# for f in csv_files:
#     print(f"Loading {os.path.basename(f)}...")
#     df = load_and_sample(f)
#     dfs.append(df)
#     print(f"  -> {len(df)} rows")

# # Combine
# df = pd.concat(dfs, ignore_index=True)
# print(f"\nTotal: {len(df):,} rows, {df.shape[1]} columns")

Loading DrDoS_NTP.csv...


/var/folders/_5/rd93j0_50qx24_jwt4qyyl700000gp/T/ipykernel_81227/2814291084.py:8: DtypeWarning: Columns (0: SimillarHTTP) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)


  -> 3455899 rows


In [4]:
# Basic info
print("Columns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes.value_counts())

Columns:
['Unnamed: 0', 'Flow ID', ' Source IP', ' Source Port', ' Destination IP', ' Destination Port', ' Protocol', ' Timestamp', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Varian

## Source IP Distribution

In [ ]:
# Source IP distribution
label_counts = df[' Source IP'].value_counts()
print("Source IP distribution:")
print(label_counts)
print(f"\nUnique IPs: {df[' Source IP'].nunique()}")

Label distribution:
 Source IP
172.16.0.5        3454063
192.168.50.4          517
192.168.50.9          381
192.168.50.8          338
192.168.50.6          330
                   ...   
52.85.89.143            1
205.185.216.10          1
34.198.228.49           1
172.217.2.98            1
172.217.1.2             1
Name: count, Length: 86, dtype: int64

Unique labels: 86


In [ ]:
# Plot label distribution
plt.figure(figsize=(12, 6))
label_counts.plot(kind='bar')
plt.title('Attack Type Distribution')
plt.xlabel('Label')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('reports/figures/label_distribution.png', dpi=150)
plt.show()

## Key Features: Duration, Packet Count, Header Size

In [ ]:
# Focus features (from CICFlowMeter)
KEY_FEATURES = [
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Fwd Header Length',
    'Bwd Header Length',
    'Protocol',
    'Flow Bytes/s',
    'Flow Packets/s'
]

# Check which exist
available_features = [f for f in KEY_FEATURES if f in df.columns]
print(f"Available key features: {len(available_features)}")
for f in available_features:
    print(f"  - {f}")

In [ ]:
# Summary stats for key features
print(df[available_features].describe())

In [ ]:
# Boxplots by attack type
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

features_to_plot = ['Flow Duration', 'Total Fwd Packets', 'Protocol', 'Flow Bytes/s']
for ax, feat in zip(axes.flatten(), features_to_plot):
    if feat in df.columns:
        df.boxplot(column=feat, by='Label', ax=ax)
        ax.set_title(feat)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.suptitle('')
plt.tight_layout()
plt.savefig('reports/figures/features_boxplot.png', dpi=150)
plt.show()

## Save Processed Data

In [ ]:
# Save sampled data
output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)
df.to_csv(f"{output_dir}/ddos_sampled.csv", index=False)
print(f"Saved {len(df):,} rows to {output_dir}/ddos_sampled.csv")